In [1]:
# import model configs
import torch
from torch.utils.data import DataLoader
import torch.nn.functional as F
from pathlib import Path
import pandas as pd
import numpy as np
from src.paths import *
from src.nano_maker.container.configs import *
from src.nano_maker.skeleton import Skeleton
from src.nano_maker.nanomaker import NanoMaker
from src.nano_maker.modules.data_handling.radial_dataset import RadialDataset
import math # angstroms and angle error

from tqdm import tqdm

In [2]:
# Loss Curve Plotting

In [3]:
RADIAL_SEQUENCES = pd.read_pickle(DATABASE / "radial_seq_df.pkl")
RADIAL_SEQUENCES.set_index("PDB_ID", inplace=True)

In [10]:
MOLECULAR_FINGERPRINTS = pd.read_pickle(DATABASE / "scaffold_molfp_df.pkl")
MOLECULAR_FINGERPRINTS.set_index("smiles_str", inplace=True)

In [11]:
test_pointer_path = DATABASE / "skeleton_test_pointers_dict.parquet"

In [12]:
# Skeleton - Iteration 1 Final val Loss was 0.232 but was a custom weighted function
# 0.5 went to radius, 0.25 went to the other two angles
# after doing some math, at WORST radius error is around 0.68 Angstroms
# but angle loss (worst case scenario) its around 62 degrees which is significant especially for
# polar, which operates on the 180 scale
# Basically I need to roughly see the actual error between the two
sk_cfg = skeleton_config
block_size = sk_cfg['block_size']
max_angstroms = sk_cfg['max_angstroms']
test_set = RadialDataset(pointer_path=test_pointer_path,
                         smiles_molfp=MOLECULAR_FINGERPRINTS,
                         pdb_radial=RADIAL_SEQUENCES,
                         block_size=block_size,
                         max_angstroms=max_angstroms)
test_loader = DataLoader(
    test_set,
    batch_size=256,
    shuffle=True,
)

In [13]:
test_model = Skeleton(n_embd=sk_cfg['n_embd'],
                      n_head=sk_cfg['n_head'],
                      n_layers=sk_cfg['n_layers'],
                      block_size=sk_cfg['block_size'],
                      map4_res=sk_cfg['map4_res'],
                      max_angstroms=sk_cfg['max_angstroms'],
                      dropout=sk_cfg['dropout'])
skeleton_weight_filename = skeleton_weight_filename
skeleton_settings = torch.load(CONTAINER / skeleton_weight_filename, map_location='cpu')
test_model.load_state_dict(skeleton_settings['model_state_dict'])
test_model.eval()


Skeleton(
  (c_project): Linear(in_features=3, out_features=512, bias=True)
  (pos_emb): Embedding(64, 512)
  (mol_encoder): MolecularEncoder(
    (block1): Sequential(
      (0): Linear(in_features=1024, out_features=1024, bias=True)
      (1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (2): GELU(approximate='none')
      (3): Dropout(p=0.15, inplace=False)
    )
    (check1): Linear(in_features=1024, out_features=512, bias=True)
    (block2): Sequential(
      (0): Linear(in_features=512, out_features=512, bias=True)
      (1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (2): GELU(approximate='none')
      (3): Dropout(p=0.15, inplace=False)
    )
    (check2): Linear(in_features=512, out_features=512, bias=True)
    (residual1): Linear(in_features=1024, out_features=512, bias=True)
    (block3): Sequential(
      (0): Linear(in_features=512, out_features=512, bias=True)
      (1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (2): GELU

In [10]:
# Analyze Protein Pocket Biochemistry as a proxy for potential zero-shot capability or model accuracy
# We expect LogP of a given drug to correlate with low hydrophobicity of a given protein pocket
# polar drug = hydrophilic protein pocket
# ok won't be able to run this until i finish naanobot I3
# 20 drug structures of varying LogPs -> 10 protein pockets generated for each
# per drug structure's LogP -> get mean protein pocket polarity
# plot

nano_maker = NanoMaker(skeleton_weight_filename, skeleton_config, naanobot_weight_filename, naanobot_config, radial_config)

In [32]:
d1 = ("CC(=O)OC1=CC=CC=C1C(=O)O", 1.18)   # aspirin
d2 = ("C1=CC(=C(C(=C1I)C=O)I)I", 3.3)
d3 = ("CN1C=NC2=C1C(=O)N(C(=O)N2C)C", -0.1)  # caffeine
d4 = ()